<a href="https://colab.research.google.com/github/woodstock0513/-20262R0136COSE34102/blob/main/kuccthon_clean_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 사투리 서비스 백엔드 (Colab)

위에서 아래로 순서대로 실행하세요. (1~4번 셀이 필수, 이후는 선택 유틸리티)

**사전 준비**
- Colab 왼쪽 🔑 *보안 비밀*에 `SUPABASE_URL`, `SUPABASE_KEY`, `NGROK_AUTHTOKEN` 등록
- `dialect_fastapi_starter.zip` 파일 준비

**제공 API**

| 경로 | 설명 |
|---|---|
| `POST /dna/analyze` | 사투리 DNA 분석 (결과를 Supabase에 저장, 음성은 저장 안 함) |
| `POST /voice-brain` | 말투 유사도 + 뇌 반응 예측 (AI 서버 프록시) |
| `GET /voice-brain/status` | AI 서버 상태 확인 |

## 1. 설치

In [ ]:
import subprocess
from google.colab import files

uploaded = files.upload()
zip_name = next((n for n in uploaded if n.endswith(".zip")), None)
if not zip_name:
    raise ValueError("dialect_fastapi_starter.zip 파일을 업로드하세요.")

subprocess.run(["unzip", "-o", "-q", zip_name], check=True)
subprocess.run(["pip", "-q", "install", "-r", "dialect_api/requirements.txt", "requests", "pyngrok"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
print("백엔드 설치 완료")

Saving dialect_fastapi_starter.zip to dialect_fastapi_starter.zip
백엔드 설치 완료


## 2. 설정
`AI_URL`은 AI 서버(ngrok) 주소입니다. 프론트엔드 주소가 정해지면 `FRONTEND_ORIGIN`에 입력한 뒤 3~4번 셀을 다시 실행하세요.

In [12]:
import os
import secrets
from google.colab import userdata

AI_URL = "https://789e-203-251-135-23.ngrok-free.app/"  # AI 서버 주소 (바뀌면 여기 수정)
FRONTEND_ORIGIN = "https://kuccthongroup4.vercel.app"                                    # 프론트엔드 주소 (선택)

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_KEY"] = userdata.get("SUPABASE_KEY")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")
os.environ["AI_API_URL"] = AI_URL.strip().rstrip("/")
os.environ["ADMIN_KEY"] = secrets.token_urlsafe(32)
os.environ["ALLOWED_ORIGINS"] = ",".join(filter(None, [
    "http://localhost:3000",
    "http://localhost:5173",
    FRONTEND_ORIGIN.strip().rstrip("/"),
]))
print("설정 완료")

설정 완료


In [13]:
import subprocess
import time
import requests

subprocess.run(
    ["fuser", "-k", "8000/tcp"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

server = subprocess.Popen(
    ["uvicorn", "dialect_api.app:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("/tmp/dialect_api.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

try:
    r = requests.get("http://127.0.0.1:8000/health", timeout=5)
    print("서버 상태:", r.status_code, r.text)
except requests.RequestException:
    print(open("/tmp/dialect_api.log").read()[-3000:])

서버 상태: 200 {"ok":true}


## 3. API 적용
`dialect_api/app.py` 끝에 `/dna/analyze`, `/voice-brain`, `/voice-brain/status`를 추가합니다. 여러 번 실행해도 중복되지 않습니다.

In [14]:
from pathlib import Path

MARKER = "# --- AUTO-APPENDED ENDPOINTS (do not edit below) ---"

ENDPOINTS = """
from fastapi import File, Form, UploadFile, HTTPException
import os
import subprocess
import tempfile
import requests
from pathlib import Path as _Path

NGROK_HEADERS = {"ngrok-skip-browser-warning": "true"}


def _ai_url() -> str:
    url = os.environ.get("AI_API_URL", "").rstrip("/")
    if not url:
        raise HTTPException(503, "AI 서버 주소 미설정")
    return url


def _read_upload(audio: UploadFile, max_mb: int) -> bytes:
    limit = max_mb * 1024 * 1024
    raw = audio.file.read(limit + 1)
    if not raw or len(raw) > limit:
        raise HTTPException(413, f"음성 파일은 {max_mb}MB 이하만 가능합니다")
    return raw


def _to_wav(raw: bytes, workdir: str, timeout: int) -> _Path:
    source = _Path(workdir) / "input_audio"
    wav = _Path(workdir) / "converted.wav"
    source.write_bytes(raw)
    try:
        subprocess.run(
            ["ffmpeg", "-nostdin", "-y", "-loglevel", "error",
             "-i", str(source), "-t", "60", "-ar", "16000", "-ac", "1",
             "-c:a", "pcm_s16le", str(wav)],
            check=True, capture_output=True, timeout=timeout,
        )
    except (subprocess.SubprocessError, OSError):
        raise HTTPException(400, "음성 파일 변환 실패")
    return wav


@app.post("/dna/analyze")
def analyze_dialect(audio: UploadFile = File(...), session_id: str = Form(...)):
    if not 8 <= len(session_id) <= 100:
        raise HTTPException(400, "session_id 형식 오류")

    raw = _read_upload(audio, 3)
    ai_url = _ai_url()

    with tempfile.TemporaryDirectory() as tmp:
        wav = _to_wav(raw, tmp, timeout=30)
        try:
            with wav.open("rb") as f:
                response = requests.post(
                    ai_url + "/predict",
                    files={"audio": ("recording.wav", f, "audio/wav")},
                    headers=NGROK_HEADERS,
                    timeout=180,
                )
            response.raise_for_status()
            results = response.json()["results"]
            if not results:
                raise ValueError("AI 응답에 결과 없음")
            best = max(results, key=lambda item: float(item["prob"]))
            scores = {item["name"]: float(item["prob"]) for item in results}
            confidence = float(best["prob"])
        except (requests.RequestException, ValueError, KeyError, TypeError):
            raise HTTPException(502, "AI 서버 응답 오류")

    saved = db.table("dna_analyses").insert({
        "session_id": session_id,
        "predicted_region": best["name"],
        "confidence": confidence,
        "scores": scores,
        "model_version": "wav2vec-bert-korean-dialect-recognition",
    }).execute().data[0]

    return {
        "analysis_id": saved["id"],
        "predicted_region": best["name"],
        "confidence": confidence,
        "scores": scores,
        "retry_recommended": confidence < 0.50,   # 50% 미만이면 재녹음 권장
        "donation_eligible": confidence >= 0.90,
    }


@app.post("/voice-brain")
def voice_brain_proxy(audio: UploadFile = File(...)):
    raw = _read_upload(audio, 10)
    ai_url = _ai_url()

    with tempfile.TemporaryDirectory() as tmp:
        wav = _to_wav(raw, tmp, timeout=40)
        try:
            with wav.open("rb") as f:
                response = requests.post(
                    ai_url + "/voice-brain",
                    files={"audio": ("audio.wav", f, "audio/wav")},
                    headers=NGROK_HEADERS,
                    timeout=300,
                )
            if response.status_code in (400, 502, 503):
                try:
                    detail = response.json().get("detail", response.text)
                except ValueError:
                    detail = response.text[:300]
                raise HTTPException(response.status_code, str(detail))
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            raise HTTPException(502, "AI 말투 분석 서버 연결 실패")


@app.get("/voice-brain/status")
def voice_brain_status():
    ai_url = _ai_url()
    try:
        response = requests.get(ai_url + "/voice-brain/status", headers=NGROK_HEADERS, timeout=15)
        response.raise_for_status()
        return response.json()
    except (requests.RequestException, ValueError):
        raise HTTPException(503, "말투 분석 서버 상태 확인 실패")
"""

app_path = Path("dialect_api/app.py")
code_text = app_path.read_text(encoding="utf-8")

# 이전에 추가된 엔드포인트가 있으면 제거 후 다시 추가
for cut in (MARKER, '@app.post("/dna/analyze")'):
    if cut in code_text:
        code_text = code_text[:code_text.index(cut)]
        break

app_path.write_text(code_text.rstrip() + "\n\n" + MARKER + "\n" + ENDPOINTS, encoding="utf-8")
print("API 적용 완료: /dna/analyze, /voice-brain, /voice-brain/status")

API 적용 완료: /dna/analyze, /voice-brain, /voice-brain/status


## 4. 서버 실행 및 외부 주소 발급
출력되는 **백엔드 주소를 프론트엔드 담당자에게 전달**하세요.

In [16]:
import os
import subprocess
import time
import requests
from pathlib import Path
from pyngrok import ngrok

LOG_PATH = "/tmp/dialect_api.log"
server = None


def start_server():
    """8000 포트의 기존 서버를 종료하고 FastAPI를 새로 실행합니다."""
    global server
    subprocess.run(["fuser", "-k", "8000/tcp"], check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
    server = subprocess.Popen(
        ["uvicorn", "dialect_api.app:app", "--host", "0.0.0.0", "--port", "8000"],
        env=os.environ.copy(),
        stdout=open(LOG_PATH, "w"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(20):
        try:
            r = requests.get("http://127.0.0.1:8000/health", timeout=3)
            r.raise_for_status()
            print("FastAPI 실행 성공:", r.json())
            return
        except requests.RequestException:
            time.sleep(1)
    print(Path(LOG_PATH).read_text()[-3000:])
    raise RuntimeError("FastAPI 실행 실패")


def get_backend_url():

    existing = next((t for t in ngrok.get_tunnels()
                     if str(t.config.get("addr", "")).endswith(":8000")), None)
    return existing.public_url if existing else ngrok.connect(8000, "http").public_url


start_server()
backend_url = get_backend_url()

print("\n백엔드      :", backend_url)
print("DNA 분석 API:", backend_url + "/dna/analyze")
print("API 문서    :", backend_url + "/docs")

FastAPI 실행 성공: {'ok': True}

백엔드      : https://gleeful-backlog-justifier.ngrok-free.dev
DNA 분석 API: https://gleeful-backlog-justifier.ngrok-free.dev/dna/analyze
API 문서    : https://gleeful-backlog-justifier.ngrok-free.dev/docs


## 유틸리티 (선택)

### AI 서버 주소 변경
새 AI 서버 주소를 입력하면 환경변수를 바꾸고 서버를 재시작합니다. (ngrok 백엔드 주소는 유지됩니다)

In [ ]:
NEW_AI_URL = "https://xxxx.ngrok-free.app"   # 새 AI 서버 주소

os.environ["AI_API_URL"] = NEW_AI_URL.strip().rstrip("/")
start_server()
print("AI 서버 주소:", os.environ["AI_API_URL"])

### 백엔드 / AI 서버 연결 확인

In [19]:
ai_url = os.environ["AI_API_URL"].rstrip("/")
targets = [
    ("백엔드(로컬)", "http://127.0.0.1:8000/health"),
    ("AI /openapi.json", ai_url + "/openapi.json"),
    ("AI /voice-brain/status", ai_url + "/voice-brain/status"),
]
for name, url in targets:
    try:
        r = requests.get(url, headers={"ngrok-skip-browser-warning": "true"}, timeout=20)
        print(f"[{r.status_code}] {name}: {r.text[:200]}")
    except requests.RequestException as e:
        print(f"[실패] {name}: {e!r}")

[200] 백엔드(로컬): {"ok":true}
[200] AI /openapi.json: {"openapi":"3.1.0","info":{"title":"사투리 DNA 모델 서버","description":"음성(wav)을 받아 사투리 지역을 추정하고(`/predict`), 같은 음성에 대한 평균적인 청자의 대뇌 피질 반응을 TRIBE v2로 예측해 기준 말투와 비교합니다(`/voice-brain`).\n\n- 요청: `multipart/for
[200] AI /voice-brain/status: {"ready":true,"refs":["무뚝뚝한 말투","밝고 친근한 말투","차분한 말투","화난 말투"]}


### 테스트: DNA 분석
음성 파일을 업로드해 `/dna/analyze`를 호출합니다. 결과만 DB에 저장됩니다.

In [ ]:
import uuid
import mimetypes
from google.colab import files

up = files.upload()
fname = next(iter(up))

r = requests.post(
    "http://127.0.0.1:8000/dna/analyze",
    data={"session_id": str(uuid.uuid4())},
    files={"audio": (fname, up[fname], mimetypes.guess_type(fname)[0] or "application/octet-stream")},
    timeout=240,
)
print("상태 코드:", r.status_code)
print(r.json() if r.ok else r.text[:1000])

Saving 20sec.m4a to 20sec.m4a
상태 코드: 200
{'analysis_id': '78a34be9-c7f5-49c1-b8b2-34a4c723feb0', 'predicted_region': '경상도', 'confidence': 0.27441462874412537, 'scores': {'경상도': 0.27441462874412537, '전라도': 0.1869218796491623, '제주도': 0.18684029579162598, '서울·경기': 0.18591046333312988, '충청도': 0.11428295820951462, '강원도': 0.0516296848654747}, 'retry_recommended': True, 'donation_eligible': False}


### 테스트: 말투·뇌 반응 분석

In [ ]:
from google.colab import files

up = files.upload()
fname = next(iter(up))

r = requests.post(
    "http://127.0.0.1:8000/voice-brain",
    files={"audio": (fname, up[fname])},
    timeout=360,
)
print("상태 코드:", r.status_code)

if r.ok:
    res = r.json()
    print("분석 길이:", res.get("seconds"), "초 / 처리 시간:", res.get("elapsed"), "초")
    print("\n[말투 유사도 순위]")
    for item in res.get("similar", []):
        print(f"  {item['style']}: {item['r']:.4f}")
    print("\n[AI 예측 뇌 영역별 반응]")
    for item in res.get("regions", []):
        print(f"  {item['name']}: {item['user']}")
    print("\n사투리 예측:", res.get("dialect"))
else:
    print("오류:", r.text[:1000])

Saving 20sec.m4a to 20sec (1).m4a
상태 코드: 200
분석 길이: 23 초 / 처리 시간: 42.3 초

[말투 유사도 순위]
  밝고 친근한 말투: 0.8679
  무뚝뚝한 말투: 0.8665
  화난 말투: 0.6881
  차분한 말투: 0.6794

[AI 예측 뇌 영역별 반응]
  청각 영역: 0.02966972626745701
  오른쪽 상측두: -0.002104321261867881
  왼쪽 상측두: -0.04348330944776535
  왼쪽 하전두: -0.02481354959309101

사투리 예측: {'label': 'Gyeongsang-do', 'name': '경상도', 'prob': 0.27441462874412537}


### 서버 로그 확인

In [ ]:
print("실행 중인 서버:")
subprocess.run(["pgrep", "-af", "uvicorn"])
print("\n최근 로그:")
print(Path("/tmp/dialect_api.log").read_text()[-4000:])

실행 중인 서버:

최근 로그:
INFO:     Started server process [4707]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:42348 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:52054 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:33948 - "POST /dna/analyze HTTP/1.1" 200 OK
INFO:     43.201.150.30:0 - "POST /dna/analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:34646 - "POST /voice-brain HTTP/1.1" 200 OK



In [ ]:
from pathlib import Path

code = Path("/content/dialect_api/app.py").read_text(encoding="utf-8")

print("CORS 설정:", "CORSMiddleware" in code)
print("허용된 출처:", __import__("os").environ.get("ALLOWED_ORIGINS"))

CORS 설정: True
허용된 출처: http://localhost:3000,http://localhost:5173


### 종료

In [ ]:
ngrok.kill()
subprocess.run(["fuser", "-k", "8000/tcp"], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("ngrok 및 서버 종료 완료")